# ImplementForge -- FreshCart Under Pressure
## COSE 278: Implementing Systems -- Day 4

**Run cells in order. Do not skip cells.**


## ⚙️ CELL 0 -- Setup
*Run first. Do not modify.*


In [ ]:
GITHUB_BASE = "https://raw.githubusercontent.com/rickghome/278game/main"
import urllib.request, os, pickle

for filename in ["if_engine.py", "if_cards.py"]:
    urllib.request.urlretrieve(f"{GITHUB_BASE}/{filename}", filename)
    print(f"  ✅  {filename} loaded")

exec(open("if_engine.py").read())
exec(open("if_cards.py").read())
print("\n✅  ImplementForge loaded.")


## 💾 CELL 0b -- Save / Restore
*Run second. Do not modify.*


In [ ]:
def save_game(game):
    with open('game_state.pkl','wb') as f: pickle.dump(game,f)
    print(f"💾  Saved -- {game['team_name']}  |  Income: ${sum(game['income'].values()):,}  |  Trust: {game['trust_score']}/100")

def restore_game():
    if not os.path.exists('game_state.pkl'):
        print("❌  No saved game. Start from CELL 1."); return None
    with open('game_state.pkl','rb') as f: game = pickle.load(f)
    print(f"✅  Restored -- {game['team_name']}  |  Income: ${sum(game['income'].values()):,}  |  Trust: {game['trust_score']}/100")
    print(f"    Seeds: {game['seeds']}")
    return game

def _run_phase(phase_cards, phase_key, decisions_list, rationales_list, game):
    import random
    env = game["environment"]
    idx = ENV_IDX[env]
    vel = game.get("velocity_multiplier", 1.0)
    phase_income = int(_baseline(env) * vel)

    if not phase_cards:
        print("✅  No incidents this phase.")
        update_income(game, phase_key, phase_income)
        print_income_summary(game)
        save_game(game)
        return

    active   = [c for c in phase_cards if c not in ["L1_positive","L3_positive"]]
    positive = [c for c in phase_cards if c in ["L1_positive","L3_positive"]]

    for cid in positive:
        card = get_card(cid, game)
        protected_loss = card["income_loss"][idx]
        phase_income -= protected_loss
        game["cards_fired"].append(cid)
        print_positive(card["name"], card["message"], protected_loss,
                       "unchanged -- your investment paid off")

    if not active:
        update_income(game, phase_key, phase_income)
        print_income_summary(game)
        save_game(game)
        return

    if len(decisions_list) < len(active):
        print(f"❌  {len(active)} card(s) fired but only {len(decisions_list)} decision(s) entered.")
        print(f"    decisions = {['a'] * len(active)}  <- add one entry per card")
        return
    if len(rationales_list) < len(active):
        print(f"❌  {len(active)} rationale(s) required.")
        return
    for i,r in enumerate(rationales_list[:len(active)]):
        if not r or r.startswith("Enter"):
            print(f"❌  rationales[{i}] required."); return
    for i,d in enumerate(decisions_list[:len(active)]):
        if d not in ["a","b","c"]:
            print(f"❌  decisions[{i}] must be a, b, or c."); return

    for i, cid in enumerate(active):
        card = get_card(cid, game)
        decision = decisions_list[i]
        rationale = rationales_list[i]
        game["cards_fired"].append(cid)
        game["decisions"][cid] = decision
        game["rationales"][cid] = rationale

        if cid == "L1" and decision == "b":
            ok = random.random() > 0.5
            key = "b_success" if ok else "b_fail"
            print("🎲  Hotfix:", "WORKED" if ok else "FAILED")
            loss_triplet = card["income_loss"].get(key,(0,0,0))
            trust_delta  = card["trust_delta"].get(key,0)
        else:
            loss_triplet = card["income_loss"].get(decision,(0,0,0))
            trust_delta  = card["trust_delta"].get(decision,0)

        income_loss = loss_triplet[idx]

        if cid == "L3":
            income_loss += card.get("income_loss_base",{}).get("locked",(0,0,0))[idx]

        mod = card.get("environment_modifier",{}).get(env,1.0)
        income_loss = int(income_loss * mod)

        if phase_key == "phase3":
            seed_count = count_seeds(game)
            if seed_count >= 2 and i == 0:
                income_loss = int(income_loss * 1.35)

        recovery = card.get("trust_recovery_modifier",{}).get(decision,0)
        trust_delta += recovery

        seed = card.get("seeds",{}).get(decision)
        if seed: plant_seed(game, seed)

        phase_income -= income_loss
        update_trust(game, trust_delta)

        add_trace(game, cid, f"{phase_key}: {cid}",
                  seed_trigger=", ".join(game["seeds"]) if game["seeds"] else None,
                  severity="minor" if income_loss < 100_000 else "major",
                  next_risk=seed)
        game["facilitator_trace"][-1]["phase"] = phase_key

        print_consequence(card["name"], decision, income_loss, trust_delta,
                          consequence_note=card.get("consequence_notes",{}).get(decision,""),
                          seed_planted=seed)

    update_income(game, phase_key, max(0, phase_income))
    print_income_summary(game)
    save_game(game)

print("💾  Save/restore + phase engine ready.")


## 🏷️ CELL 1 -- Team Setup


In [ ]:
TEAM_NAME = "Team Name Here"   # <- change this
game = new_game_state(TEAM_NAME)
print(f"✅  Team: {TEAM_NAME}  |  Trust: {game['trust_score']}/100")


## 🏗️ FRAME 1 -- Architecture Config

You designed FreshCart in COSE 260. Now you're building it.
**Choose one value per field. Run the cell when done.**


In [ ]:
system_profile = {

    "environment": "consumer",
    # consumer    -- fast market, high volume, volatile trust
    # enterprise  -- contract-driven, stable, reputational risk
    # government  -- fixed budget, procurement rules, political exposure

    "team_structure": "stream_aligned",
    # stream_aligned  -- small teams, end-to-end ownership, low coordination cost
    # platform        -- shared services team supporting others
    # siloed          -- functional departments, high handoff cost

    "build_buy_configure": "build",
    # build       -- we write it ourselves
    # buy         -- we purchase a product
    # configure   -- we implement a vendor platform

    "primary_risk": "delivery_speed",
    # delivery_speed   -- we might be too slow
    # technical_debt   -- we might cut corners
    # integration      -- our components might not fit together
    # vendor_lock      -- we might become too dependent on third parties
    # team_capability  -- we might not have the right skills

    "data_architecture": "shared_db",
    # dedicated_dbs  -- each service owns its data -- higher cost, clean boundaries
    # shared_db      -- single shared database -- lower cost, faster to build

    "architecture_pattern": "monolith",
    # monolith      -- single deployable unit -- simple, fast, coupling debt deferred
    # layered       -- structured layers -- small overhead, coupling risk over time
    # client_server -- central server -- familiar, bottleneck risk at scale
    # event_driven  -- async messaging -- scalable, hard to debug without tooling
    # microservices -- independent services -- flexible, high coordination cost upfront

    "coupling": "medium",   # fixed -- do not change
}

is_valid, errors, warnings = validate_frame1(system_profile)
if errors:
    print("❌  Fix these errors:")
    for e in errors: print(f"    {e}")
else:
    game["frame1"] = system_profile
    game["environment"] = system_profile["environment"]
    print("✅  Frame 1 accepted.")
    for w in warnings: print(f"  {w}")
    print(f"\n    Environment:      {system_profile['environment']}")
    print(f"    Team:             {system_profile['team_structure']}")
    print(f"    Strategy:         {system_profile['build_buy_configure']}")
    print(f"    Risk:             {system_profile['primary_risk']}")
    print(f"    Data:             {system_profile['data_architecture']}")
    print(f"    Architecture:     {system_profile['architecture_pattern']}")
    apply_architecture_tax(game)
    save_game(game)


## ⚠️ PHASE 0 -- Architecture Stress Test

Your architecture choice just met its first real constraint.
One card fires -- determined by your pattern choice.
No config prevents it. Only your decision determines the damage.


In [ ]:
_arch_cards = select_architecture_cards(system_profile, game)
if not _arch_cards:
    print("✅  No architecture stress this phase.")
else:
    card = get_card(_arch_cards[0], game)
    game["cards_fired"].append(_arch_cards[0])
    print_card(card["name"], card["scenario"], card["options"], card["flavor"])
    print("\n    Enter decision in next cell.")


In [ ]:
decisions  = ["a"]
rationales = ["Enter one sentence explaining why you chose this option"]
_run_phase(_arch_cards, "phase1", decisions, rationales, game)
print("\n    Scroll to FRAME 2.")


## 🔧 FRAME 2 -- Pipeline Config

You have seen Phase 0. Now configure your pipeline.
**Engineering Capacity hard cap: 100 units.**


In [ ]:
# +-----------------------------------------+----------+----------+
# |  Choice                                 |  Cap cost|  Speed   |
# +-----------------------------------------+----------+----------|
# |  testing: minimal                       |     5    |  fast    |
# |  testing: standard                      |    15    |  moderate|
# |  testing: thorough                      |    30    |  slow    |
# |  deploy:  big_bang                      |     5    |  fastest |
# |  deploy:  rolling                       |    15    |  moderate|
# |  deploy:  blue_green                    |    25    |  moderate|
# |  deploy:  canary                        |    30    |  slowest |
# |  rollback: none                         |     0    |  --       |
# |  rollback: partial                      |    10    |  --       |
# |  rollback: full                         |    20    |  --       |
# |  observability: none                    |     0    |  --       |
# |  observability: basic                   |    10    |  --       |
# |  observability: full                    |    25    |  --       |
# |  change_owner: single_person            |     0    |  --       |
# |  change_owner: shared_pair              |    10    |  --       |
# |  change_owner: team_owned               |    20    |  --       |
# +-----------------------------------------+----------+----------+

pipeline_profile = {
    "testing_coverage":    "standard",    # minimal | standard | thorough
    "deployment_method":   "rolling",     # big_bang | rolling | blue_green | canary
    "rollback_plan":       "partial",     # none | partial | full
    "observability_level": "basic",       # none | basic | full
    "change_owner":        "shared_pair", # single_person | shared_pair | team_owned
}

is_valid, errors, warnings, capacity_used = validate_frame2(pipeline_profile)
print(f"Engineering Capacity: {capacity_used}/100 units\n")
if errors:
    print("❌  Fix these errors:")
    for e in errors: print(f"    {e}")
else:
    game["frame2"] = pipeline_profile
    print("✅  Frame 2 accepted.")
    for w in warnings: print(f"  {w}")
    print(f"\n    Testing:       {pipeline_profile['testing_coverage']}")
    print(f"    Deployment:    {pipeline_profile['deployment_method']}")
    print(f"    Rollback:      {pipeline_profile['rollback_plan']}")
    print(f"    Observability: {pipeline_profile['observability_level']}")
    print(f"    Change owner:  {pipeline_profile['change_owner']}")
    save_game(game)
    print("\n    Scroll to Phase 1 -- Build.")


## ⚠️ PHASE 1 -- Build

FreshCart is under construction. Architecture and org choices meeting reality.


In [ ]:
# PHASE 1 show cards
_phase1_cards = select_build_cards(system_profile, game)
__phase1_cards_active   = [c for c in _phase1_cards if c not in ["L1_positive","L3_positive"]]
__phase1_cards_positive = [c for c in _phase1_cards if c in  ["L1_positive","L3_positive"]]

for cid in __phase1_cards_positive:
    card = get_card(cid, game)
    print_positive(card["name"], card["message"],
                   card["income_loss"][ENV_IDX[game["environment"]]], "unchanged")

if not __phase1_cards_active:
    if not __phase1_cards_positive:
        print("No incidents this phase.")
else:
    n = len(__phase1_cards_active)
    print("PHASE 1 -- " + str(n) + " CARD(S)")
    print("-" * 50)
    for i,cid in enumerate(__phase1_cards_active):
        card = get_card(cid, game)
        print("CARD " + str(i+1) + " of " + str(n) + ": " + card["name"])
        print()
        print_card(card["name"], card["scenario"], card["options"], card["flavor"])
        if card.get("timed"):
            print("TIMED -- " + str(card["timer_seconds"]) + " seconds. No entry = option (b).")
        print()
    print("Enter decisions in next cell.")


In [ ]:
# _phase1_cards -- decisions
decisions  = ["a"]
rationales = ["Enter one sentence explaining why you chose this option"]
_run_phase(_phase1_cards, "phase1", decisions, rationales, game)
print("\n    Scroll to Frame 3.")


## 🛠️ FRAME 3 -- Operations Config

You have seen Phases 0 and 1. Configure your operational layer.


In [ ]:
operations_profile = {
    "vendor_dependency":  "medium",         # low | medium | high
    "fallback_strategy":  "manual",         # none | manual | automated
    "on_call_coverage":   "business_hours", # none | business_hours | follow_the_sun | full_24x7
    "incident_response":  "runbook",        # ad_hoc | runbook | practiced
}

is_valid, errors, warnings = validate_frame3(operations_profile)
if errors:
    print("❌  Fix these errors:")
    for e in errors: print(f"    {e}")
else:
    game["frame3"] = operations_profile
    print("✅  Frame 3 accepted.")
    for w in warnings: print(f"  {w}")
    print(f"\n    Vendor:    {operations_profile['vendor_dependency']}")
    print(f"    Fallback:  {operations_profile['fallback_strategy']}")
    print(f"    On-call:   {operations_profile['on_call_coverage']}")
    print(f"    Incidents: {operations_profile['incident_response']}")
    save_game(game)
    print("\n    Scroll to Phase 2a -- Pipeline Run.")


## ⚠️ PIPELINE PHASE -- Gate Decisions

Your change moves from development toward production.
At each gate you decide how much to invest.
**What you skip here becomes a production incident later.**

Gates 1-4 are config decisions. Gate 5 is your go/no-go.


In [ ]:
# PIPELINE GATE CONFIG
# Fill out all five gates. Run when done.

pipeline_gates = {

    # GATE 1 -- UNIT TEST
    # Who: Developer. Question: does this piece work?
    "unit_test_strategy": "full",
    # full        -- all code paths including edge cases
    # happy_path  -- main flows only, skip edge cases
    # skip        -- no unit tests

    # GATE 2 -- INTEGRATION TEST
    # Who: Dev + QA. Question: do the pieces work together?
    "integration_owner": "dev_and_qa",
    # dev_and_qa  -- joint ownership, catches cross-team assumptions
    # dev_only    -- fast, misses what QA would catch
    # skip        -- nothing tested together

    "integration_scope": "full",
    # full          -- all integration points
    # critical_path -- main flows only
    # skip          -- not tested

    # GATE 3 -- STAGING
    # Who: QA + Ops. Question: does it work in a production-like environment?
    "staging_fidelity": "production_mirror",
    # production_mirror -- same data volume, config, network topology
    # partial_mirror    -- similar config, smaller data volume
    # dev_extended      -- staging is just dev with a different name

    # GATE 4 -- UAT
    # Who: BUSINESS USERS -- not IT
    # Question: does it do what the business needs?
    "uat_owner": "business_users",
    # business_users -- real users, real workflows, real questions
    # it_team        -- faster, but answers the wrong question
    # skip           -- no user validation before go-live

    "uat_coverage": "full_workflows",
    # full_workflows -- all business processes tested
    # happy_path     -- main journeys only
    # skip           -- no coverage

    # GATE 5 -- GO / NO-GO
    # Your call. Based on what the pipeline found.
    "go_nogo_decision": "go",
    # go              -- ship as planned
    # conditional_go  -- ship with documented known issues
    # no_go           -- delay -- fix first
}

is_valid, errors, warnings = validate_pipeline_gates(pipeline_gates)

if errors:
    print("Fix these errors:")
    for e in errors: print(f"  {e}")
else:
    game["pipeline_gates"] = pipeline_gates
    for w in warnings: print(w)
    print()

    # Calculate costs and seeds
    total_cost, gate_seeds, bugs_caught, cost_breakdown = calculate_pipeline_costs(
        pipeline_gates, game["environment"])

    # Plant gate seeds
    for s in gate_seeds:
        plant_seed(game, s)

    # Deduct pipeline cost from phase income
    env = game["environment"]
    vel = game.get("velocity_multiplier", 1.0)
    phase_income = int(_baseline(env) * vel) - total_cost
    update_income(game, "phase2", max(0, phase_income))

    # Print cost curve
    print_pipeline_cost_curve(pipeline_gates, cost_breakdown, bugs_caught,
                               gate_seeds, env)

    # Print go/no-go assessment
    print_go_nogo_assessment(pipeline_gates, gate_seeds, game)

    save_game(game)
    print("Scroll to Phase 3 -- Live.")


## ⚠️ PHASE 3 -- Live

FreshCart is live. Real load. Real users. Real consequences.

TIMED️ **One card runs under a 60-second timer.**


In [ ]:
# PHASE 3 show cards
_live_cards = select_live_cards(system_profile, pipeline_profile, game)
__live_cards_active   = [c for c in _live_cards if c not in ["L1_positive","L3_positive"]]
__live_cards_positive = [c for c in _live_cards if c in  ["L1_positive","L3_positive"]]

for cid in __live_cards_positive:
    card = get_card(cid, game)
    print_positive(card["name"], card["message"],
                   card["income_loss"][ENV_IDX[game["environment"]]], "unchanged")

if not __live_cards_active:
    if not __live_cards_positive:
        print("No incidents this phase.")
else:
    n = len(__live_cards_active)
    print("PHASE 3 -- " + str(n) + " CARD(S)")
    print("-" * 50)
    for i,cid in enumerate(__live_cards_active):
        card = get_card(cid, game)
        print("CARD " + str(i+1) + " of " + str(n) + ": " + card["name"])
        print()
        print_card(card["name"], card["scenario"], card["options"], card["flavor"])
        if card.get("timed"):
            print("TIMED -- " + str(card["timer_seconds"]) + " seconds. No entry = option (b).")
        print()
    print("Enter decisions in next cell.")


In [ ]:
# _live_cards -- decisions
decisions  = ["a"]
rationales = ["Enter one sentence explaining why you chose this option"]
_run_phase(_live_cards, "phase3", decisions, rationales, game)
print("\n    Scroll to Phase 4 -- v2 Release.")


## ⚠️ PHASE 4 -- v2 Release

**Universal requirement -- same for every team:**

> FreshCart is adding real-time delivery tracking.
> New Tracking Service, updated mobile API, change to Notification Service.
> All teams implement this. What happens depends on what you built.


In [ ]:
_v2_cards = select_v2_cards(system_profile, pipeline_profile, game)
_v2_active = [c for c in _v2_cards if c not in ["L1_positive","L3_positive"]]
_seed_count = count_seeds(game)

if _seed_count >= 2:
    print(f"⚠  Entering Phase 4 with {_seed_count} active seeds. Severity escalated.\n")

if not _v2_active:
    print("✅  No major incidents this phase. Earlier decisions held.")
    update_income(game, "phase4", int(_baseline(game["environment"]) * game.get("velocity_multiplier",1.0)))
    print_income_summary(game)
    save_game(game)
else:
    print(f"{'-'*50}\nPHASE 4 -- {len(_v2_active)} CARD(S)\n{'-'*50}\n")
    for i,cid in enumerate(_v2_active):
        card = get_card(cid, game)
        print(f"--- CARD {i+1} of {len(_v2_active)}: {card['name']} ---\n")
        print_card(card["name"], card["scenario"], card["options"], card["flavor"])
        print()
    print("    Enter decisions in next cell.")


In [ ]:
decisions  = ["a"]
rationales = ["Enter one sentence explaining why you chose this option"]
_run_phase(_v2_cards, "phase4", decisions, rationales, game)
print("\n    Scroll to Phase 5 -- Scale Event.")


## 🚨 PHASE 5 -- Scale Event

**Wait for your instructor to read the scenario aloud.**

No new config. Your architecture holds -- or it doesn't.


In [ ]:
def _calculate_s1_outcome(game):
    env   = game["environment"]
    idx   = ENV_IDX[env]
    base  = int(PHASE_BASELINE[idx] * game.get("velocity_multiplier",1.0))
    seeds = game.get("seeds",[])
    f2    = game.get("frame2",{})
    f3    = game.get("frame3",{})
    score = 0
    deploy = f2.get("deployment_method","big_bang")
    if deploy in ["canary","blue_green"]: score += 2
    elif deploy == "rolling":             score += 1
    rollback = f2.get("rollback_plan","none")
    if rollback == "full":     score += 2
    elif rollback == "partial": score += 1
    obs = f2.get("observability_level","none")
    if obs == "full":    score += 2
    elif obs == "basic": score += 1
    fallback = f3.get("fallback_strategy","none") if f3 else "none"
    if fallback == "automated": score += 2
    elif fallback == "manual":  score += 1
    ir = f3.get("incident_response","ad_hoc") if f3 else "ad_hoc"
    if ir == "practiced": score += 2
    elif ir == "runbook":  score += 1
    effective = score - len(seeds) * 0.5
    is_gov = env == "government"
    if effective >= 7:
        tier, income, td = "thriving", base + int(base*0.68), 10
        narrative = (f"Traffic spiked 35x. Systems flagged it in 90 seconds.\n"
                     f"Automated fallback activated. FreshCart degraded gracefully.\n\n"
                     f"Revenue captured: +${int(base*0.68):,} above baseline. Trust: +10.\n\nThis is what you paid for.")
    elif effective >= 4:
        tier, income, td = "surviving", int(base*0.95), 0
        narrative = "Traffic spiked 35x. FreshCart bent but didn't break. Most revenue captured."
    elif effective >= 1:
        tier, income, td = "struggling", int(base*0.6), -15
        narrative = "Traffic spiked 35x. FreshCart struggled. Major revenue loss. Trust eroded."
    else:
        tier, income, td = "collapsing", int(base*0.2), -30
        narrative = "Traffic spiked 35x. FreshCart collapsed. The accumulated decisions produced this outcome."
    if is_gov: income = max(income, int(base*0.4))
    seed_notes, extra_loss, extra_trust = [], 0, 0
    seed_map = {
        "P1_silent_fraud":    ((400_000,300_000,240_000), -40, "P1 silent fraud surfaces."),
        "V2_mock_data":       ((200_000,150_000,120_000), -35, "V2 mock data discovered."),
        "D1_db_scaling":      ((150_000,112_000, 90_000), -10, "DB saturation triggered."),
        "P3_workaround_live": ((120_000, 90_000, 72_000), -10, "P3 config cascade."),
        "V1_queue_risk":      ((180_000,135_000,108_000), -25, "V1 queue overflow."),
        "P2_shared_ownership":((100_000, 75_000, 60_000), -15, "P2 ownership failure."),
        "PT1_integration_debt":((80_000,  60_000, 48_000), -10, "PT1 integration debt surfaces."),
        "PT2_env_mismatch":   ((60_000,  45_000, 36_000),  -5, "PT2 env mismatch in production."),
        "PT3_uat_skip":       ((90_000,  67_000, 54_000), -15, "PT3 UAT bug found by customers."),
        "PT4_race_condition": ((70_000,  52_000, 42_000), -10, "PT4 race condition triggered."),
        "PT5_prod_config_risk":((80_000, 60_000, 48_000),  -8, "PT5 prod config failure."),
        "A1_coupling_debt":   ((120_000, 90_000, 72_000), -10, "A1 monolith coupling collapses."),
        "A2_layer_debt":      ((80_000,  60_000, 48_000),  -8, "A2 layer violations compound."),
        "A3_scale_cliff":     ((200_000,150_000,120_000), -20, "A3 client-server hits scale cliff."),
        "A4_trace_gap":       ((90_000,  67_000, 54_000), -10, "A4 async failures untraceable."),
        "A5_stakeholder_doubt":((60_000, 45_000, 36_000), -15, "A5 stakeholder confidence lost."),
    }
    for seed_id, (losses, trust_hit, note) in seed_map.items():
        if seed_id in seeds:
            extra_loss += losses[idx]; extra_trust += trust_hit; seed_notes.append(note)
    income = max(0, income - extra_loss)
    td += extra_trust
    if is_gov: income = max(income, int(base*0.4))
    return tier, income, td, narrative, seed_notes

_tier, _s1_income, _s1_trust, _s1_narrative, _seed_resolutions = _calculate_s1_outcome(game)
game["cards_fired"].append("S1")
print(f"\n{'-'*50}\nPHASE 5 -- THE MOMENT OF TRUTH\n{'-'*50}")
print(_s1_narrative)
if _seed_resolutions:
    print("\nSeeds resolving:")
    for s in _seed_resolutions: print(f"  💥 {s}")
update_income(game, "phase5", _s1_income)
update_trust(game, _s1_trust)
print(f"\nOutcome tier: {_tier.upper()}")
print_income_summary(game)
save_game(game)
print("\n    Scroll to S2 -- The Bill Arrives.")


In [ ]:
print(f"\n{'-'*50}\nTHE BILL ARRIVES\n{'-'*50}")
print(f"\nYour Trust Score: {game['trust_score']}/100")
env_label = {"consumer":"Customer Trust","enterprise":"Stakeholder Confidence",
             "government":"Political Capital"}.get(game["environment"],"Trust")
print(f"({env_label})")
print("\n... room discussion -- 30 seconds ...\n")
print("Run next cell to reveal revenue and enter decision.")


In [ ]:
_card_s2 = get_card("S2", game)
game["cards_fired"].append("S2")
print_card(_card_s2["name"], _card_s2["scenario"], _card_s2["options"], _card_s2["flavor"])


In [ ]:
decision_rationale_s2 = "Enter one sentence explaining why you chose this option"
decision_s2 = "a"

if decision_rationale_s2.startswith("Enter"):
    print("❌  decision_rationale_s2 required.")
elif decision_s2 not in ["a","b","c"]:
    print("❌  decision must be a, b, or c.")
else:
    game["decisions"]["S2"] = decision_s2
    game["rationales"]["S2"] = decision_rationale_s2
    env = game["environment"]
    idx = ENV_IDX[env]
    loss = _card_s2["income_loss"].get(decision_s2,(0,0,0))[idx]
    td   = _card_s2["trust_delta"].get(decision_s2,0)
    rec  = _card_s2.get("trust_recovery_modifier",{}).get(decision_s2,0)
    seed = _card_s2.get("seeds",{}).get(decision_s2)
    if seed: plant_seed(game, seed)
    game["income"]["phase5"] = max(0, game["income"].get("phase5",0) - loss)
    update_trust(game, td)
    if rec: print(f"    Trust recovery: +{rec}")
    print_consequence(_card_s2["name"], decision_s2, loss, td,
                      consequence_note=_card_s2.get("consequence_notes",{}).get(decision_s2,""),
                      seed_planted=seed)
    save_game(game)
    print("\n    Government teams: check next cell for S3.")
    print("    All others: scroll to Final Summary.")


### Government teams only
*Skip if not government.*


In [ ]:
if game.get("frame1",{}).get("environment") != "government":
    print("✅  Not government. Skip to Final Summary.")
elif not _should_fire_S3(game.get("frame1",{}), game):
    print("✅  S3 did not fire.")
else:
    _card_s3 = get_card("S3", game)
    game["cards_fired"].append("S3")
    print_card(_card_s3["name"], _card_s3["scenario"], _card_s3["options"], _card_s3["flavor"])


In [ ]:
decision_rationale_s3 = "Enter one sentence explaining why you chose this option"
decision_s3 = "a"

if game.get("frame1",{}).get("environment") != "government":
    print("✅  Skip.")
elif not _should_fire_S3(game.get("frame1",{}), game):
    print("✅  S3 did not fire.")
elif decision_rationale_s3.startswith("Enter"):
    print("❌  decision_rationale_s3 required.")
else:
    _card_s3 = get_card("S3", game)
    game["decisions"]["S3"] = decision_s3
    game["rationales"]["S3"] = decision_rationale_s3
    env = game["environment"]
    idx = ENV_IDX[env]
    loss = _card_s3["income_loss"].get(decision_s3,(0,0,0))[idx]
    td   = _card_s3["trust_delta"].get(decision_s3,0)
    game["income"]["phase5"] = max(0, game["income"].get("phase5",0) - loss)
    update_trust(game, td)
    print_consequence(_card_s3["name"], decision_s3, loss, td,
                      consequence_note=_card_s3.get("consequence_notes",{}).get(decision_s3,""))
    save_game(game)


## 🔍 PHASE 6 -- Post-Mortem

The game is over. Now the learning begins.

Two questions. Answer honestly.
This is not blame. This is the system finding its own failure mode.


In [ ]:
# POST-MORTEM CONFIG
# Fill out all three fields. Run when done.

postmortem = {

    "assumption_that_failed": "speed_over_quality",
    # speed_over_quality          -- we optimized for delivery speed
    # team_knows_the_system       -- we assumed tribal knowledge was sufficient
    # vendor_will_deliver         -- we trusted the vendor roadmap
    # staging_mirrors_prod        -- we assumed staging was good enough
    # users_will_adapt            -- we assumed users would figure it out
    # debt_can_wait               -- we assumed we could pay down debt later
    # architecture_will_scale     -- we assumed the architecture would hold
    # pipeline_gates_are_optional -- we assumed gates were bureaucracy

    "earliest_catch_point": "unit_test_gate",
    # architecture_decision    -- Frame 1 config
    # team_structure_choice    -- org design
    # unit_test_gate           -- Gate 1
    # integration_gate         -- Gate 2
    # staging_gate             -- Gate 3
    # uat_gate                 -- Gate 4
    # go_nogo_decision         -- Gate 5
    # phase3_live_incident     -- first production failure
    # phase4_v2_release        -- v2 compounding

    "one_thing_different": "Describe one concrete change you would make before starting again",
}

if postmortem["one_thing_different"].startswith("Describe"):
    print("Fill out all three fields before running.")
else:
    game["postmortem"] = postmortem
    print_postmortem_analysis(postmortem, game)
    save_game(game)
    print("Scroll to Reflection -- second iteration.")


## 📊 Final Summary
*Copy output and give to your instructor.*


In [ ]:
print_final_summary(game)


## 🔁 Reflection

**Fill this out before starting the second iteration.**


In [ ]:
strategy_change = {
    "what_failed":    "Describe the main thing that went wrong",
    "what_we_change": "Describe the specific config change you are making",
    "why":            "One sentence: why do you believe this change fixes the problem",
}

if any(v.startswith("Describe") for v in strategy_change.values()):
    print("❌  Fill out all three fields.")
else:
    game["strategy_change"] = strategy_change
    print("Reflection recorded:")
    for k,v in strategy_change.items(): print(f"  {k}: {v}")
    save_game(game)
    print("\nRun next cell to begin second iteration.")


## 🔄 Second Iteration Setup


In [ ]:
import copy
if not game.get("strategy_change") or any(v.startswith("Describe") for v in game.get("strategy_change",{}).values()):
    print("❌  Complete the reflection first.")
else:
    game_v1 = copy.deepcopy(game)
    game["income"]            = {"phase1":0,"phase2":0,"phase3":0,"phase4":0,"phase5":0}
    game["trust_score"]       = 100
    game["cards_fired"]       = []
    game["decisions"]         = {}
    game["rationales"]        = {}
    game["facilitator_trace"] = []
    game["seeds"]             = []
    game["phase"]             = 0
    game["iteration"]         = 2
    print("✅  Second iteration ready.")
    print("    Scroll back to FRAME 1, update config, run all cells again.")
    print("\n    When done: Final Summary, then Iteration Comparison.")


## 📊 Iteration Comparison
*Run after completing both iterations.*


In [ ]:
try:
    print_iteration_comparison(game_v1, game)
except NameError:
    print("❌  Complete both iterations before comparing.")
